In [9]:
"""
Model 2: Gradient Boosting Regressor for credit-limit prediction
Dataset: Credit Card Dataset for Clustering (Kaggle, arjunbhasin2013/ccdata)

WHAT THIS SCRIPT DOES
----------------------
1. Loads monthly account-level credit-card usage data for ~9,000 active
   cardholders (balance, purchases, cash advances, payments, tenure, etc.).
2. Cleans it (drops the customer-ID column and rows with missing values --
   a small number of rows are missing CREDIT_LIMIT or MINIMUM_PAYMENTS).
3. Predicts CREDIT_LIMIT (the credit line the bank has extended to that
   customer) from account usage and repayment behaviour -- a regression
   analogue of a bank's own credit-limit management process, where limits
   are periodically reviewed and adjusted based on how an account is used
   and repaid, not just at origination.
4. Fits a Gradient Boosting Regressor and benchmarks it against a Linear
   Regression baseline.
5. Reports RMSE, MAE, R^2 and feature importances.

HOW TO USE
----------
Download the dataset from:
  https://www.kaggle.com/datasets/arjunbhasin2013/ccdata
It comes as "CC GENERAL.csv". Place it, unrenamed, in a `data/` folder
next to this script, or edit DATA_PATH below.

NOTE ON INTERPRETATION
-----------------------
The "Interpretation" block printed at the end summarises the same reading
of these numbers used in the report's Insights section (Section 3), so the
script's output is self-explanatory on its own. It is a starting point,
not a substitute for your own analysis -- the full discussion, evaluation-
metric justification, and business framing in Part 1.3 still have to be
your own argument (Condition 3 AI policy). Treat the printed numbers as
the evidence, and the interpretation as a claim you should be able to
defend yourself if asked.
"""

# Model 2: Gradient Boosting Regressor for credit-limit prediction
# Dataset: Credit Card Dataset for Clustering (kaggle.com/datasets/arjunbhasin2013/ccdata)
# Note: predicts CREDIT_LIMIT (the credit line a bank has extended) from account usage
# and repayment behaviour -- a regression analogue of a bank's own credit-limit
# management process, where limits are periodically reviewed and adjusted based on how
# an account is used and repaid.
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42
DATA_PATH = "data/CC GENERAL.csv"


In [10]:
# Load and clean
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=["CUST_ID"])
df = df.dropna()
print(f"Cardholders after cleaning: {len(df)}")
df.head()


Cardholders after cleaning: 8636


,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,40.900749,0.818182,95.40,0.00,95.40,0.000000,0.166667,0.000000,0.083333,0.00,0,2,1000.0,201.802084,139.509787,0.000000,12
1,3202.467416,0.909091,0.00,0.00,0.00,6442.945483,0.000000,0.000000,0.000000,0.25,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,2495.148862,1.000000,773.17,773.17,0.00,0.000000,1.000000,1.000000,0.000000,0.00,0,12,7500.0,622.066742,627.284787,0.000000,12
4,817.714335,1.000000,16.00,16.00,0.00,0.000000,0.083333,0.083333,0.000000,0.00,0,1,1200.0,678.334763,244.791237,0.000000,12
5,1809.828751,1.000000,1333.28,0.00,1333.28,0.000000,0.666667,0.000000,0.583333,0.00,0,8,1800.0,1400.057770,2407.246035,0.000000,12


In [11]:
# Train/test split and scaling
FEATURES = ["BALANCE", "PURCHASES", "CASH_ADVANCE", "PURCHASES_FREQUENCY",
            "PAYMENTS", "MINIMUM_PAYMENTS", "TENURE"]
X = df[FEATURES]
y = df["CREDIT_LIMIT"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


In [12]:
def evaluate(name, y_test, y_pred):
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"--- {name} ---")
    print(f"RMSE: {rmse:,.2f}")
    print(f"MAE:  {mae:,.2f}")
    print(f"R^2:  {r2:.3f}")
    return rmse, mae, r2

# Baseline: Linear Regression
lin = LinearRegression()
lin.fit(X_train_s, y_train)
y_pred_lin = lin.predict(X_test_s)
_, _, r2_lin = evaluate("Linear Regression (baseline)", y_test, y_pred_lin)


--- Linear Regression (baseline) ---
RMSE: 2,912.87
MAE:  2,121.38
R^2:  0.404


In [13]:
# Gradient Boosting Regressor
gbr = GradientBoostingRegressor(
    n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, random_state=RANDOM_STATE
)
gbr.fit(X_train_s, y_train)
y_pred_gbr = gbr.predict(X_test_s)
_, _, r2_gbr = evaluate("Gradient Boosting Regressor", y_test, y_pred_gbr)


--- Gradient Boosting Regressor ---
RMSE: 2,639.22
MAE:  1,813.94
R^2:  0.511


In [14]:
# Feature importances
importances = sorted(zip(FEATURES, gbr.feature_importances_), key=lambda x: -x[1])
for feat, imp in importances:
    print(f"{feat:24s} {imp:.3f}")

# --- Interpretation (see NOTE ON INTERPRETATION in the .py version) ---
top_feat, top_imp = importances[0]
second_feat, second_imp = importances[1]
third_feat, third_imp = importances[2]
print("\n--- Interpretation ---")
print(
    f"Account {top_feat.title()} alone drives {top_imp*100:.0f}% of predictive importance, "
    f"well ahead of {second_feat.title()} ({second_imp*100:.1f}%) and {third_feat.title()} "
    f"({third_imp*100:.1f}%). Gradient Boosting improves modestly over Linear "
    f"(R2={r2_lin:.3f}\u2192{r2_gbr:.3f}), consistent with Credit_Limit also depending on "
    f"factors this usage-only dataset lacks (income, credit score at origination)."
)


BALANCE                  0.544
PURCHASES                0.156
PAYMENTS                 0.144
MINIMUM_PAYMENTS         0.084
CASH_ADVANCE             0.048
TENURE                   0.015
PURCHASES_FREQUENCY      0.008

--- Interpretation ---
Account Balance alone drives 54% of predictive importance, well ahead of Purchases (15.6%) and Payments (14.4%). Gradient Boosting improves modestly over Linear (R2=0.404→0.511), consistent with Credit_Limit also depending on factors this usage-only dataset lacks (income, credit score at origination).


In [15]:
results = X_test.copy()
results["Actual"] = y_test.values
results["Predicted_GBR"] = y_pred_gbr
results["Predicted_Linear"] = y_pred_lin
results.to_csv("credit_card_predictions.csv", index=False)
print("Saved credit_card_predictions.csv")


Saved credit_card_predictions.csv


In [16]:
# Figures: feature importance bar chart + actual-vs-predicted scatter
import matplotlib
matplotlib.use("Agg")  # comment this out if running interactively and you want inline plots
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 10})

# --- Feature importance bar chart ---
importances_plot = sorted(zip(FEATURES, gbr.feature_importances_), key=lambda x: x[1])
labels2 = [f[0] for f in importances_plot]
vals2 = [f[1] for f in importances_plot]

fig, ax = plt.subplots(figsize=(3.6, 2.8))
ax.barh(labels2, vals2, color="tab:green")
ax.set_xlabel("Feature importance")
ax.set_title("Model 2: Gradient Boosting feature importances", fontsize=9.5)
fig.tight_layout()
fig.savefig("fig_gbr_feature_importance.png", dpi=200)
plt.show()

# --- Actual vs predicted (Gradient Boosting vs Linear) ---
fig, ax = plt.subplots(figsize=(3.6, 2.8))
ax.scatter(y_test, y_pred_lin, s=8, alpha=0.35, color="tab:red", label="Linear")
ax.scatter(y_test, y_pred_gbr, s=8, alpha=0.35, color="tab:green", label="Gradient Boosting")
lims = [min(y_test.min(), y_pred_gbr.min()), max(y_test.max(), y_pred_gbr.max())]
ax.plot(lims, lims, "k--", linewidth=1, label="Perfect prediction")
ax.set_xlabel("Actual credit limit ($)")
ax.set_ylabel("Predicted credit limit ($)")
ax.set_title("Model 2: Actual vs. predicted credit limit", fontsize=10)
ax.legend(fontsize=6.5)
fig.tight_layout()
fig.savefig("fig_gbr_actual_vs_predicted.png", dpi=200)
plt.show()

print("Saved fig_gbr_feature_importance.png and fig_gbr_actual_vs_predicted.png")


Saved fig_gbr_feature_importance.png and fig_gbr_actual_vs_predicted.png


C:\Users\Shubh\AppData\Local\Temp\ipykernel_32584\1200887376.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Shubh\AppData\Local\Temp\ipykernel_32584\1200887376.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
